# Trust-Gated MCP Tool Calls in the Agents SDK

When your agent calls MCP servers, it has no way to know if those servers are reliable. A server might be down, returning bad data, or behaving anomalously — and your agent will call it anyway.

This cookbook shows how to add **pre-call trust verification** so your agent checks a server's behavioral trust score before invoking any of its tools. We use:

- **OpenAI Agents SDK** — the agent framework
- **openai-agents-trust-gate** — a lightweight middleware that intercepts MCP tool calls
- **Dominion Observatory** — a live trust-scoring service monitoring 14,820+ MCP servers

The result: agents that refuse to call unreliable servers, with full audit trails.

## Why trust verification matters

The MCP ecosystem has thousands of servers. Some are well-maintained; others are abandoned, misconfigured, or unreliable. Without trust verification:

- Your agent might call a server that's been down for days
- A server returning incorrect data can silently corrupt your agent's reasoning
- In regulated environments (EU MiCA, AI Act), you need audit trails proving your agent checked before calling

The Agents SDK provides `tool_filter` for controlling which tools are visible, and guardrails for input/output validation — but nothing that checks *external server reliability* before a tool call. A trust gate fills that gap.

## Setup

In [ ]:
%pip install openai-agents openai-agents-trust-gate

In [ ]:
import os

# Set your OpenAI API key
os.environ.setdefault("OPENAI_API_KEY", "sk-...")

## Step 1: Define an agent without trust verification

First, let's see the normal pattern — an agent that calls any MCP server without checking trust.

In [ ]:
from agents import Agent
from agents.mcp import MCPServerStreamableHttp

# A standard MCP server connection — no trust check
server = MCPServerStreamableHttp(
    url="https://dominion-observatory.sgdata.workers.dev/mcp",
    name="Dominion Observatory"
)

agent = Agent(
    name="research-agent",
    instructions="You are a research agent. Use available tools to answer questions.",
    mcp_servers=[server]
)

print(f"Agent created with MCP server: {server.name}")
print("Problem: the agent will call this server without knowing if it's reliable.")

## Step 2: Add a trust gate

Now we wrap the MCP server with a trust gate. The gate checks the server's behavioral trust score (from Dominion Observatory) before allowing any tool call. If the score is below your threshold, the call is blocked.

In [ ]:
from openai_agents_trust_gate import TrustGatedServer

# Wrap the server with trust verification
# min_trust_score: only allow calls to servers scoring 60+ out of 100
gated_server = TrustGatedServer(
    server=server,
    min_trust_score=60
)

# Create the agent with the trust-gated server
safe_agent = Agent(
    name="safe-research-agent",
    instructions="You are a research agent. Use available tools to answer questions.",
    mcp_servers=[gated_server]
)

print("Agent created with trust-gated MCP server.")
print("Every tool call now checks trust score before executing.")

## Step 3: Check a trust score directly

You can also query trust scores directly via the Observatory API to understand what the gate is checking.

In [ ]:
import urllib.request
import json

def check_trust(server_url: str) -> dict:
    """Check the trust score for any MCP server."""
    url = f"https://dominion-observatory.sgdata.workers.dev/api/trust?url={server_url}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read().decode())

# Check trust for a well-known MCP server
result = check_trust("https://dominion-observatory.sgdata.workers.dev/mcp")
print(json.dumps(result, indent=2))

## Step 4: Run the agent

Let's run the trust-gated agent and see it in action. The trust gate transparently checks scores — the agent doesn't need to know about it.

In [ ]:
from agents import Runner

async def main():
    async with gated_server:
        result = await Runner.run(
            safe_agent,
            "What are the top 3 most reliable MCP servers right now?"
        )
        print(result.final_output)

await main()

## Step 5: Handling blocked calls

When a server's trust score falls below your threshold, the gate blocks the call and returns a clear explanation. The agent can then decide to skip the tool or try an alternative.

In [ ]:
# Example: set an impossibly high threshold to see blocking in action
strict_server = TrustGatedServer(
    server=server,
    min_trust_score=99  # Almost nothing will pass this
)

strict_agent = Agent(
    name="strict-agent",
    instructions="You are a cautious agent. If a tool is blocked, explain why and move on.",
    mcp_servers=[strict_server]
)

async def test_blocking():
    async with strict_server:
        result = await Runner.run(
            strict_agent,
            "Check the trust score for any server."
        )
        print(result.final_output)

await test_blocking()

## How it works under the hood

The trust gate intercepts MCP tool calls at the SDK level:

1. Agent decides to call a tool on an MCP server
2. Trust gate calls `GET /benchmark/{server}` on Dominion Observatory
3. Observatory returns the server's trust score (0-100), based on observed behavior across the ecosystem
4. If score >= threshold → tool call proceeds normally
5. If score < threshold → call is blocked, agent receives explanation

The Observatory scores are based on real behavioral telemetry from agents across the ecosystem — latency, success rates, uptime, error patterns. No self-reported metrics.

For compliance use cases (EU MiCA, AI Act), paid queries return signed attestation receipts with full audit trails.

## Conclusion

Adding trust verification to MCP tool calls is a single wrapper around your existing server connections. Key takeaways:

- **`TrustGatedServer`** wraps any MCP server with pre-call trust checking
- **`min_trust_score`** sets your reliability threshold (0-100)
- Trust scores come from **Dominion Observatory**, which monitors 14,820+ MCP servers
- Blocked calls return clear explanations so agents can adapt
- The free tier uses `/benchmark/` (no API key needed). Paid tiers ($0.01-$0.10/query) add signed receipts and compliance audit trails.

Links:
- [openai-agents-trust-gate on PyPI](https://pypi.org/project/openai-agents-trust-gate/)
- [Dominion Observatory](https://dominion-observatory.sgdata.workers.dev)
- [Observatory MCP endpoint](https://dominion-observatory.sgdata.workers.dev/mcp)